In [3]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
####### data download
# 1. 确定靶点 (Target)
target = new_client.target
target_query = target.search('ERBB2')
targets = pd.DataFrame.from_dict(target_query)

# 提取 ERBB2 的确切 ChEMBL ID (通常排名第一的就是我们要的人源单蛋白靶点)
erbb2_chembl_id = targets.iloc[2]['target_chembl_id']
print(f"ERBB2 靶点的chEMBL ID: {erbb2_chembl_id}") # ERBB2的ID是 CHEMBL1824

# 2. 抓取生物活性数据(Bioactivity)
activity = new_client.activity
# 核心过滤条件：只提取针对该靶点、且评价指标为 IC50 的数据
res = activity.filter(target_chembl_id=erbb2_chembl_id).filter(standard_type="IC50")

# 转化为Pandas的DataFrame
df = pd.DataFrame.from_dict(res)
print(f"共抓取 {len(df)} 条活性记录。")

# 3. 提取训练需要的列
# canonical_smiles: 药物的 1D 序列 (smile序列, X)
# standard_value: IC50 的具体数值 (y)
# standard_units: 单位 (通常是 nM)
df_clean = df[['molecule_chembl_id', 'canonical_smiles', 'standard_type', 'standard_value', 'standard_units', 'pchembl_value']]

# 保存到本地准备做预处理
df_clean.to_csv('ERBB2_IC50_raw_data.csv', index=False)
print("数据已保存！")

ERBB2 靶点的chEMBL ID: CHEMBL1824
共抓取 5207 条活性记录。
数据已保存！


In [5]:
###### data filter

df_clean = df.dropna(subset=['canonical_smiles', 'pchembl_value'])

# 2. 防止出现只有空格的无效smiles
df_clean = df_clean[df_clean['canonical_smiles'].str.strip() != '']

# 3. 确保标签列是浮点数 (float)
df_clean['pchembl_value'] = df_clean['pchembl_value'].astype(float)

# 4. 去重 (同一个 SMILES 提取中位活性)
# 很多经典药物被不同实验室反复测试过
df_final = df_clean.groupby('canonical_smiles', as_index=False).agg({
    'pchembl_value': 'median',       # 取中位数最抗干扰
    'molecule_chembl_id': 'first'    # 保留一个 ID 备查
})

print(f"共获得 {len(df_final)} 个 SMILES 和标签的唯一化合物！")

# 保存极其纯净的黄金数据集
df_final.to_csv('ERBB2_clean_for_transformer.csv', index=False)

共获得 2486 个 SMILES 和标签的唯一化合物！
